# 03 - Tower of Hanoi（河內塔）：用遞迴拆解問題

## 學習目標

完成本 Notebook 後，你將能夠：

- 說明河內塔的三條規則。
- 把 $n$ 層問題拆成兩個 $n-1$ 層子問題。
- 寫出正確的遞迴終止條件。
- 分析 $2^n-1$ 次移動與遞迴深度。
- 正確理解 `sys.setrecursionlimit()` 能做什麼、不能做什麼。


## 1. 問題規則

有三根柱子 `A`、`B`、`C`，`A` 上放著由大到小的 $n$ 個圓盤。目標是把所有圓盤移到 `C`。

每次移動必須遵守：

1. 一次只能移動一個圓盤。
2. 只能拿走某根柱子最上面的圓盤。
3. 大圓盤不能放在小圓盤上。


## 2. 核心遞迴演算法

要把 $n$ 個圓盤從 `source` 移到 `target`，並使用 `auxiliary` 當輔助柱：

1. 先把上面的 $n-1$ 個圓盤從 `source` 移到 `auxiliary`。
2. 把最大的第 $n$ 號圓盤從 `source` 移到 `target`。
3. 再把 $n-1$ 個圓盤從 `auxiliary` 移到 `target`。

每個 $n$ 層問題都會產生兩個 $n-1$ 層問題：

$$T(n)=2T(n-1)+1$$

解開後得到：

$$T(n)=2^n-1$$

終止條件是 `n == 0`：沒有圓盤需要移動，直接返回。


In [1]:
def hanoi(
    n: int,
    source: str,
    auxiliary: str,
    target: str,
    moves: list[tuple[int, str, str]],
) -> None:
    '''將 n 個圓盤從 source 移至 target，並把步驟加入 moves。'''
    # 終止條件：沒有圓盤時，不需要進行任何操作。
    if n == 0:
        return

    # 1. 把上方 n-1 個圓盤移到輔助柱。
    hanoi(n - 1, source, target, auxiliary, moves)

    # 2. 把最大的第 n 號圓盤移到目標柱。
    moves.append((n, source, target))

    # 3. 把剛才的 n-1 個圓盤從輔助柱移到目標柱。
    hanoi(n - 1, auxiliary, source, target, moves)


def solve_hanoi(n: int) -> list[tuple[int, str, str]]:
    '''建立並回傳 n 層河內塔的完整移動步驟。'''
    if n < 0:
        raise ValueError("n 必須是非負整數")

    moves = []
    hanoi(n, "A", "B", "C", moves)
    return moves


In [2]:
# n=3 時共有 2^3-1 = 7 步
moves = solve_hanoi(3)

for step, (disk, source, target) in enumerate(moves, start=1):
    print(f"第 {step} 步：將圓盤 {disk} 從 {source} 移到 {target}")

assert len(moves) == 2**3 - 1


第 1 步：將圓盤 1 從 A 移到 C
第 2 步：將圓盤 2 從 A 移到 B
第 3 步：將圓盤 1 從 C 移到 B
第 4 步：將圓盤 3 從 A 移到 C
第 5 步：將圓盤 1 從 B 移到 A
第 6 步：將圓盤 2 從 B 移到 C
第 7 步：將圓盤 1 從 A 移到 C


## 3. 驗證每一步是否合法

寫出答案後，不只要檢查步數，也可以模擬三根柱子的狀態，確認沒有把大圓盤放到小圓盤上。


In [3]:
def verify_hanoi(n: int, moves: list[tuple[int, str, str]]) -> bool:
    # 清單尾端代表柱子的最上方。
    rods = {
        "A": list(range(n, 0, -1)),
        "B": [],
        "C": [],
    }

    for disk, source, target in moves:
        # 要移動的圓盤必須位於來源柱最上方。
        if not rods[source] or rods[source][-1] != disk:
            return False

        # 目標柱為空，或其頂端圓盤比目前圓盤大，才可以放置。
        if rods[target] and rods[target][-1] < disk:
            return False

        rods[source].pop()
        rods[target].append(disk)

    return rods["C"] == list(range(n, 0, -1))


for n in range(0, 8):
    moves = solve_hanoi(n)
    assert len(moves) == 2**n - 1
    assert verify_hanoi(n, moves)

print("n=0 到 n=7 的步驟全部合法！")


n=0 到 n=7 的步驟全部合法！


## 4. 遞迴上限與可行性

Python 為了避免無限遞迴耗盡記憶體，預設會限制遞迴深度。必要時可以調整：

```python
import sys
sys.setrecursionlimit(20000)
```

但要注意：

- 調高限制只代表允許更深的函式呼叫，不會讓演算法變快。
- 河內塔需要 $2^n-1$ 步，`n` 稍大就無法實際列出所有步驟。
- 遞迴上限設得過高可能耗盡系統堆疊；必須了解風險後再調整。


In [4]:
import sys

print("目前的遞迴上限：", sys.getrecursionlimit())

# 課堂範例：只有確定問題需要，而且系統資源足夠時才調高。
sys.setrecursionlimit(20000)
print("調整後的遞迴上限：", sys.getrecursionlimit())

# 即使不列出移動步驟，也能直接計算理論步數。
for n in [3, 10, 20, 64]:
    print(f"n={n:>2}，最少需要 {2**n - 1:,} 步")


目前的遞迴上限： 3000
調整後的遞迴上限： 20000
n= 3，最少需要 7 步
n=10，最少需要 1,023 步
n=20，最少需要 1,048,575 步
n=64，最少需要 18,446,744,073,709,551,615 步


## 5. 複雜度、常見錯誤與練習

複雜度：

- 時間複雜度：$O(2^n)$，因為必須產生 $2^n-1$ 次移動。
- 遞迴呼叫堆疊：$O(n)$。
- 若保存全部移動步驟，清單本身需要 $O(2^n)$ 空間。

常見錯誤：

- 忘記 `n == 0` 的終止條件，造成無限遞迴。
- 第二次遞迴時，把來源柱、輔助柱、目標柱的順序傳錯。
- 誤以為調高遞迴上限就能解決指數成長問題。

**練習：**修改 `hanoi()`，不要保存 `moves`，而是在移動圓盤時直接 `print()`。思考這會減少哪一部分的空間使用量？


In [5]:
def print_hanoi(n: int, source: str, auxiliary: str, target: str) -> None:
    # TODO：參考 hanoi() 的三個步驟，直接印出每一次移動。
    pass


# 完成後可取消下一行註解：
# print_hanoi(3, "A", "B", "C")
